In [1]:
# imports
import os, sys, time

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
datadir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

print("Complete!")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/valuation
Complete!


In [2]:
# load the NSI extracted data
nsi_fire = os.path.join(projdir,"data/spatial/nsi_structures_in_fires.geojson")
nsi_fire = gpd.read_file(nsi_fire)
nsi_fire.columns

Index(['fd_id', 'bid', 'occtype', 'st_damcat', 'bldgtype', 'found_type',
       'cbfips', 'pop2amu65', 'pop2amo65', 'pop2pmu65', 'pop2pmo65', 'sqft',
       'num_story', 'ftprntid', 'ftprntsrc', 'students', 'found_ht',
       'val_struct', 'val_cont', 'val_vehic', 'source', 'med_yr_blt',
       'firmzone', 'o65disable', 'u65disable', 'x', 'y', 'ground_elv',
       'ground_elv_m', 'FIPS', 'index_right', 'MTBS_ID', 'Incid_Name',
       'Ig_Date', 'geometry'],
      dtype='object')

In [3]:
# check on NAs and summary stats
# how many structures total?
print(f"Found {len(nsi_fire)} NSI structure in MTBS perimeters.")

Found 96750 NSI structure in MTBS perimeters.


In [4]:
# calculate the median value for residential and non-residential for each fire
df = nsi_fire.copy()
df['RES'] = df['st_damcat'].apply(lambda x: 'R' if x == 'RES' else 'NR')
# make sure there is valid data to play with
df = df[
    df["val_struct"].notnull() &
    (df["val_struct"] > 0)
]
# group by fire and building type (residential / non-residential), and calculate statistics
grouped = df.groupby(["Incid_Name", "MTBS_ID", "RES"], observed=True).agg(
    val_struct = ('val_struct', 'median'),
    n_struct = ('val_struct', 'count')
).reset_index()

# Pivot to wide format: one row per fire, one column per building type
res_val = grouped.pivot(index=["Incid_Name","MTBS_ID"], columns="RES", values=["val_struct","n_struct"])
res_val.columns = [f"{var}_{res}" for var, res in res_val.columns]
res_val = res_val.reset_index()

# handle NaNs
res_val['n_struct_R'] = res_val['n_struct_R'].fillna(0)
res_val['n_struct_NR'] = res_val['n_struct_NR'].fillna(0)

# create a proportion column
res_val['n_struct_total'] = res_val['n_struct_R'] + res_val['n_struct_NR']
res_val['prop_R'] = res_val['n_struct_R'] / res_val['n_struct_total']

# round the USD values
res_val['val_struct_R'] = res_val['val_struct_R'].round(2).astype("float64")
res_val['val_struct_NR'] = res_val['val_struct_NR'].round(2).astype("float64")

res_val.head()

,Incid_Name,MTBS_ID,val_struct_NR,val_struct_R,n_struct_NR,n_struct_R,n_struct_total,prop_R
0,0268 SUGARLOAF PR,OR4459311956220150627,475224.61,269595.23,1.0,4.0,5.0,0.8
1,0353 RN SCOTT CANYON,OR4552312036720160721,NaN,352411.82,0.0,2.0,2.0,1.0
2,0368 TEN MILE CANYON RN,OR4492212094420150708,NaN,184645.23,0.0,3.0,3.0,1.0
3,1026,MT4599610625820170824,896333.32,NaN,3.0,0.0,3.0,0.0
4,115 / MEERS,OK3480409861420220729,NaN,221860.11,0.0,1.0,1.0,1.0


In [5]:
# save out.
out_fp = os.path.join(projdir,"data/tabular/MTBS_NSI_summary.csv")
res_val.to_csv(out_fp, index=False)
print(f"Saved to: {out_fp}")

Saved to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/valuation/data/tabular/MTBS_NSI_summary.csv
